# Trust Game: download and preprocess one public participant

**Author:** Smith Lab  
**Updated:** 2026-08-18  
**License:** MIT

This notebook retrieves only `sub-10317` Trust data from OpenNeuro `ds005123` version `1.1.3` and runs focused fMRIPrep for runs 1 and 2.

## What you will learn

1. Freeze an OpenNeuro snapshot with DataLad.
2. Retrieve only one participant's required files.
3. Run pinned fMRIPrep and inspect its report.


## 1. Load pinned software

The first Neurodesk module load may take several minutes.

In [ ]:
import module
await module.load('fmriprep/25.2.5')
await module.list()

In [ ]:
%pip install -q nibabel matplotlib watermark
from pathlib import Path
import glob, json, subprocess
import nibabel as nib
from IPython.display import IFrame, display

## 2. Configure a workspace outside the repository

In [ ]:
SUBJECT='10317'; SESSION='01'; SNAPSHOT='1.1.3'; TASK='trust'; RUNS=('1','2')
WORKSPACE=Path.home()/'trust_teaching'; BIDS_DIR=WORKSPACE/'ds005123'; DERIVATIVES_DIR=WORKSPACE/'derivatives'; SCRATCH_DIR=WORKSPACE/'scratch'/'fmriprep'; FILTER_FILE=WORKSPACE/'bids_filters.json'
for path in (WORKSPACE,DERIVATIVES_DIR,SCRATCH_DIR): path.mkdir(parents=True,exist_ok=True)
print(f'Workspace: {WORKSPACE}')
print('Warning: focused fMRIPrep may take several hours.')

## 3. Install the frozen dataset and retrieve only required files

In [ ]:
source='https://github.com/OpenNeuroDatasets/ds005123.git'
if not (BIDS_DIR/'.datalad').exists(): subprocess.run(['datalad','install','-s',source,str(BIDS_DIR)],check=True)
subprocess.run(['git','-C',str(BIDS_DIR),'checkout',SNAPSHOT],check=True)
patterns=[f'sub-{SUBJECT}/ses-{SESSION}/anat/*',f'sub-{SUBJECT}/ses-{SESSION}/fmap/*']
for run in RUNS:
    patterns += [f'sub-{SUBJECT}/ses-{SESSION}/func/*task-{TASK}_run-{run}*_bold.nii.gz',f'sub-{SUBJECT}/ses-{SESSION}/func/*task-{TASK}_run-{run}*_bold.json',f'sub-{SUBJECT}/ses-{SESSION}/func/*task-{TASK}_run-{run}*_events.tsv',f'sub-{SUBJECT}/ses-{SESSION}/func/*task-{TASK}_run-{run}*_sbref.nii.gz',f'sub-{SUBJECT}/ses-{SESSION}/func/*task-{TASK}_run-{run}*_sbref.json']
targets=sorted({p for pattern in patterns for p in glob.glob(str(BIDS_DIR/pattern))})
if not targets: raise FileNotFoundError('No matching public Trust files found.')
subprocess.run(['datalad','get',*targets],cwd=BIDS_DIR,check=True)
print(f'Retrieved or verified {len(targets)} files.')

## 4. Check the FreeSurfer license

fMRIPrep checks for a personal license even with `--fs-no-reconall`; no license is embedded here.

In [ ]:
FS_LICENSE=Path.home()/'.license'
if not FS_LICENSE.is_file(): raise FileNotFoundError('Save your personal FreeSurfer license as ~/.license, then rerun.')
print(FS_LICENSE)

## 5. Run focused fMRIPrep

The filter prevents processing unrelated RF1 tasks. MNI 2-mm output is used for this public exercise.

In [ ]:
bids_filter={'bold':{'datatype':'func','task':TASK,'run':list(RUNS),'suffix':'bold'},'sbref':{'datatype':'func','task':TASK,'run':list(RUNS),'suffix':'sbref'},'t1w':{'datatype':'anat','suffix':'T1w'},'t2w':{'datatype':'anat','suffix':'T2w'},'fmap':{'datatype':'fmap'}}
FILTER_FILE.write_text(json.dumps(bids_filter,indent=2)+'\n')
display(bids_filter)
def combined_bold(run):
    pattern=f'sub-{SUBJECT}/ses-{SESSION}/func/*task-{TASK}_run-{run}*space-MNI152NLin6Asym*desc-preproc_bold.nii.gz'
    return [p for p in DERIVATIVES_DIR.glob(pattern) if 'echo-' not in p.name]
if all(len(combined_bold(run))==1 for run in RUNS): print('Complete outputs exist; skipping fMRIPrep.')
else:
    command=['fmriprep',str(BIDS_DIR),str(DERIVATIVES_DIR),'participant','--participant-label',SUBJECT,'--session-label',SESSION,'--bids-filter-file',str(FILTER_FILE),'--output-spaces','MNI152NLin6Asym:res-2','--me-output-echos','--fs-no-reconall','--fs-license-file',str(FS_LICENSE),'--nprocs','8','--omp-nthreads','2','--mem-mb','16000','--stop-on-first-crash','-w',str(SCRATCH_DIR)]
    print('Running:',' '.join(command)); subprocess.run(command,check=True)

## 6. Inspect outputs and basic QC

Review every report section, especially distortion correction, registration, carpet plots, and confounds.

In [ ]:
report=DERIVATIVES_DIR/f'sub-{SUBJECT}.html'
if not report.is_file(): raise FileNotFoundError(report)
display(IFrame(src=str(report),width='100%',height=700))
for run in RUNS:
    candidates=combined_bold(run)
    if len(candidates)!=1: raise RuntimeError(f'Expected one run-{run} combined BOLD: {candidates}')
    img=nib.load(candidates[0]); print(f'run-{run}: shape={img.shape}; voxel={img.header.get_zooms()[:3]}')

## 7. Dependencies and handoff

Notebook 02 uses these frozen canonical events and fMRIPrep outputs, then calls the repository production EV and L1 scripts for both runs.

In [ ]:
%load_ext watermark
%watermark
%watermark --iversions
await module.list()